In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import re
import json
import orjson
import pandas as pd
import numpy as np

from functools import partial
from sparsify import Sae
from sparsify.data import chunk_and_tokenize
from sparsify import SparseCoder

from delphi.config import CacheConfig, ConstructorConfig, RunConfig, SamplerConfig
from delphi.sparse_coders import load_hooks_sparse_coders
from delphi.latents import LatentCache, LatentDataset
from delphi.latents.collect_activations import collect_activations
from delphi.explainers import DefaultExplainer
from delphi.clients import Offline, OpenRouter
from delphi.pipeline import Pipe, Pipeline, process_wrapper

from delphi.scorers import FuzzingScorer, DetectionScorer
from delphi.explainers import explanation_loader,random_explanation_loader

import torch
from torch.utils.data import DataLoader
from safetensors.numpy import load_file, save_file
from dataclasses import asdict
from pathlib import Path
from openai import OpenAI
from datasets import load_dataset, load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from models.modeling_qwen3_moe import Qwen3MoeForCausalLM

from tqdm import tqdm

In [2]:
def topk_sae_for_experts(activations_expert, activations_sae, k=5):
    # 转换为 DataFrame
    df_exp = pd.DataFrame(activations_expert, columns=["sample", "token", "expert"])
    df_sae = pd.DataFrame(activations_sae, columns=["sample", "token", "sae_unit"])
    
    # 按 (sample, token) 合并
    df = pd.merge(df_exp, df_sae, on=["sample", "token"])
    
    # 统计 (expert, sae_unit) 共现次数
    counts = df.value_counts(["expert", "sae_unit"]).reset_index(name="count")
    
    # 按 expert 分组取 top-k
    topk = counts.groupby("expert").apply(
        lambda x: x.nlargest(k, "count")
    ).reset_index(drop=True)
    
    return topk


In [7]:
activations_expert = np.array([
    [0, 0, 1],
    [0, 1, 1],
    [0, 1, 2],
    [1, 0, 2],
])

activations_sae = np.array([
    [0, 0, 100],
    [0, 1, 100],
    [0, 1, 101],
    [1, 0, 200],
    [1, 0, 201],
])

res = topk_sae_for_experts(activations_expert, activations_sae, k=3)

/tmp/ipykernel_2347447/2388833138.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  topk = counts.groupby("expert").apply(


In [8]:
res

,expert,sae_unit,count
0,1,100,2
1,1,101,1
2,2,100,1
3,2,101,1
4,2,200,1


# compute detection performance of each neuron

In [2]:
folder_path = "outputs/e5-base-v2/sae/evaluations/sentence_embedding"
results = []

# 遍历文件夹中的所有 json 文件
for file in os.listdir(folder_path):

    match = re.search(r"latent(\d+)", file)
    latent_id = int(match.group(1))

    file_path = os.path.join(folder_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # data 是一个列表，每个元素是一个样本的预测结果
    total = len(data)

    total_positive = sum(1 for sample in data if sample['activating'])
    correct = sum(1 for sample in data if sample['correct'])
    accuracy = correct / total if total > 0 else 0.0

    results.append({
        "latent id": latent_id,
        "total": total,
        "total positives": total_positive,
        "correct": correct,
        "accuracy": accuracy
    })

# 保存到 CSV
df = pd.DataFrame(results)
output_file = f"{folder_path}.csv"
df.to_csv(output_file, index=False)

print(f"结果已保存到 {output_file}")

结果已保存到 outputs/e5-base-v2/sae/evaluations/sentence_embedding.csv


# transform SAE weight format

In [10]:
weight_new = load_file('./expert-0-layer-47/global_step_90000.safetensors')

In [109]:
weight_new = torch.load('sparse_ckpts/Qwen3-30B-A3B-Instruct-2507/transcoder_3060.pt', weights_only=True)

In [81]:
weight_new.keys()

dict_keys(['input_bias', 'encoder.weight', 'encoder.bias', 'decoder.weight', 'decoder.bias'])

In [117]:
# weight_new_ = {
#     'encoder.weight': weight_new['module.encoder.weight'],
#     'encoder.bias': weight_new['module.encoder.bias'],
#     'W_dec': weight_new['module.decoder.weight'].T,
#     'b_dec': weight_new['module.decoder.bias'],
# }

weight_new_ = {
    'encoder.weight': weight_new['encoder.weight'].numpy(),
    'encoder.bias': weight_new['encoder.bias'].numpy(),
    'W_dec': weight_new['encoder.weight'].numpy().T,
    'b_dec': np.zeros([2048], dtype=np.float32),
}
save_path = "sparse_ckpts/Qwen3-30B-A3B-Instruct-2507/transcoder/layers.47.mlp.gate"
os.makedirs(save_path, exist_ok=True)
save_file(weight_new_, f'{save_path}/sae.safetensors')

In [124]:
weight = load_file('./sparse_ckpts/Qwen3-30B-A3B-Instruct-2507/transcoder/layers.47.mlp.gate/sae.safetensors')

In [126]:
weight['encoder.weight'].dtype

dtype('float32')

In [19]:
weight_new['module.decoder.bias'].shape

(2048,)

In [1]:
def call_llm(prompt):
    client = OpenAI(
        api_key='bigai',
        base_url=f"http://115.190.119.173:8888/v1",  
        timeout=1800,
    )
    response = client.chat.completions.create(
        model="qwen3", 
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4096,
    )
    
    output = response.choices[0].message.content
    return output

In [4]:
call_llm('你好呀')

'你好呀！✨ 很高兴见到你～今天过得怎么样呀？希望你度过了愉快的一天！(•̀ᴗ•́)و'

In [3]:
latents = load_file('outputs/e5-base-v2/sae/raw_latents/sentence_embedding/train.safetensors')
tokens = latents['tokens']

In [3]:
latents.keys()

dict_keys(['activations', 'locations', 'tokens'])

In [4]:
latents['tokens'].shape

(200008, 512)

In [5]:
latents['locations'][:, 2].max()

6143

In [6]:
e5_tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-large-v2")
qwen_tokenizer = AutoTokenizer.from_pretrained("../../models/Qwen3-30B-A3B-Instruct-2507/")
qwen_tokenizer.add_special_tokens({"cls_token": "[CLS]", "pad_token": "[PAD]"})
CLS_ID = qwen_tokenizer.convert_tokens_to_ids("[CLS]")
PAD_ID = qwen_tokenizer.convert_tokens_to_ids("[PAD]")

In [16]:
input_ids = latents['tokens'][20000]
qwen_tokenizer.decode(input_ids)

'passage: Subacute thyroiditis (SAT) is an inflammatory disorder of the thyroid caused probably by viruses\nThe principal classes of viruses involed  in SAT include Epstein Barr and Retroviridae[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][

In [70]:
MAX_LEN = 512
qwen_tokenized = []

for row in tqdm(tokens):
    # 去掉 padding
    ids = row[row != 0]
    ids_wo_cls = ids[1:]  # 去掉 e5 的 [CLS]
    
    # 解码成文本
    text = e5_tokenizer.decode(ids_wo_cls, skip_special_tokens=True)
    
    # 用 Qwen3 tokenizer 编码
    qwen_ids = qwen_tokenizer.encode(text, add_special_tokens=False)
    
    # 插入 [CLS]
    qwen_ids = [CLS_ID] + qwen_ids
    
    # 截断或填充到 MAX_LEN
    if len(qwen_ids) > MAX_LEN:
        qwen_ids = qwen_ids[:MAX_LEN]
    else:
        qwen_ids = qwen_ids + [PAD_ID] * (MAX_LEN - len(qwen_ids))
    
    qwen_tokenized.append(qwen_ids)

qwen_tokenized = np.array(qwen_tokenized, dtype=np.int64)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 423116/423116 [04:47<00:00, 1472.20it/s]


In [72]:
qwen_tokenized = np.array(qwen_tokenized, dtype=np.int32)

In [75]:
latents['tokens'] = qwen_tokenized
save_file(latents, "outputs/e5-large-v2/sae/raw_latents/sentence_embedding/0_8191.safetensors")

In [56]:

qwen_tokenizer.convert_tokens_to_ids("[CLS]") 

151669

In [46]:
i = 10
valid_tokens = tokens[i]

In [47]:
tokenizer.batch_decode([valid_tokens])

['[CLS] passage : the text explains that a specific line ( ref = " b 10 " ) was found in a pbf file and visualized using qgis after filtering with osmosis. the author also mentions using osm2pgsql ( a tool for importing openstreetmap data into postgresql ) with a custom style, running on a 32 - bit postgresql 9. 3 setup on windows 7. additionally, they note that mapnik, a mapping toolkit, is currently only available for 32 - bit windows, which may cause issues with 64 - bit versions of postgis, postgresql, or python. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PA

In [31]:
latents['locations'][:, 2].max()

8191

In [17]:
np.unique(latents['locations'][:, 2])

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        41,  42,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,
        55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,
        69,  70,  71,  72,  73,  74,  76,  77,  78,  79,  80,  82,  83,
        84,  85,  86,  87,  88,  89,  90,  92,  93,  94,  95,  96,  97,
        98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
       111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123,
       124, 125, 126, 127])

In [19]:
latents['activations'].max()

1.192

In [6]:
latents = torch.randn(20, 10, 5)

In [8]:
topk_activations, topk_indices = torch.topk(latents, 3, dim=-1)

In [13]:
topk_indices.shape

torch.Size([20, 10, 3])

In [12]:
torch.nonzero(latents.abs() > 1e-5).shape

torch.Size([1000, 3])